# 02. Intent Taxonomy Discovery — Clustering & Silhouette Analysis

This notebook documents how the 7-class SpotifyCares intent taxonomy was derived through unsupervised embedding + KMeans clustering + silhouette sweeps.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

CACHE_DIR = Path("../data/cache")
df = pd.read_parquet(CACHE_DIR / "spotify_threads.parquet")
print(f"Loaded {len(df):,} Spotify thread pairs")

In [ ]:
# Sample for silhouette sweep
sample_df = df.sample(min(10_000, len(df)), random_state=42)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(sample_df["customer_msg"].tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=True)
print(f"Computed embeddings shape: {embeddings.shape}")

In [ ]:
# Evaluate k from 4 to 10
for k in range(4, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels, sample_size=2000, random_state=42)
    print(f"Clusters k={k}: Silhouette Score = {score:.4f}")